# User Story → Test Case Generator
**Stack:** LangChain + LangGraph + Ollama (llama3.2)  
**Domain:** QA / Banking / Insurance  

**Agent flow:**
```
User Story Input
      ↓
[Story Analyzer Node]  — extracts actors, actions, conditions
      ↓
[Test Case Generator Node] — generates positive, negative, edge cases
      ↓
[Reviewer Node]  — validates coverage and flags gaps
      ↓
Structured Test Cases Output (saved to .ipynb-friendly display)
```

In [1]:
%pip install -q langchain langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import json, textwrap
from typing import TypedDict, Annotated
from IPython.display import display, Markdown
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
print('Imports OK')

Imports OK


In [3]:
import urllib.request, json as _j
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
        models = [m["name"] for m in _j.loads(r.read()).get("models", [])]
        print("Ollama running. Models:", models)
        if not any("llama3.2" in m for m in models):
            print("WARNING: llama3.2 not found. Run: ollama pull llama3.2")
except Exception as e:
    print("Ollama not reachable:", e)
    print("Start Ollama from system tray or run: ollama serve")


Ollama running. Models: ['llama3.2:latest']


In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.3,
)
print("LLM ready:", llm.model)


LLM ready: llama3.2


## State Definition
LangGraph passes this dict between nodes.

In [5]:
class TCGState(TypedDict):
    user_story: str          # input
    analysis: str            # output of Story Analyzer
    test_cases: str          # output of Test Case Generator
    review: str              # output of Reviewer
    final_output: str        # formatted final result


## Node 1 — Story Analyzer

In [6]:
analyze_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior QA analyst with 20+ years in banking and insurance. "
     "Analyze the given user story and extract:\n"
     "1. Actor(s) — who is performing the action\n"
     "2. Action(s) — what they are doing\n"
     "3. Goal — the expected outcome\n"
     "4. Preconditions — what must be true before the action\n"
     "5. Business Rules — domain-specific constraints\n"
     "Be concise and structured."),
    ("human", "User Story:\n{user_story}")
])

analyze_chain = analyze_prompt | llm | StrOutputParser()

def story_analyzer_node(state: TCGState) -> TCGState:
    print("[Node 1] Analyzing user story...")
    analysis = analyze_chain.invoke({"user_story": state["user_story"]})
    return {**state, "analysis": analysis}


## Node 2 — Test Case Generator

In [7]:
tcg_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert QA test case writer specializing in banking and insurance domains. "
     "Given a user story analysis, generate comprehensive test cases.\n\n"
     "For each test case include:\n"
     "- Test Case ID (TC_001, TC_002 ...)\n"
     "- Test Case Title\n"
     "- Type (Positive / Negative / Edge / Boundary)\n"
     "- Preconditions\n"
     "- Test Steps (numbered)\n"
     "- Expected Result\n"
     "- Priority (High / Medium / Low)\n\n"
     "Generate at least 6 test cases covering positive, negative, and edge scenarios."),
    ("human",
     "User Story:\n{user_story}\n\n"
     "Analysis:\n{analysis}")
])

tcg_chain = tcg_prompt | llm | StrOutputParser()

def test_case_generator_node(state: TCGState) -> TCGState:
    print("[Node 2] Generating test cases...")
    test_cases = tcg_chain.invoke({
        "user_story": state["user_story"],
        "analysis": state["analysis"]
    })
    return {**state, "test_cases": test_cases}


## Node 3 — Reviewer

In [8]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a QA Lead reviewer. Review the generated test cases and:\n"
     "1. Confirm coverage (positive, negative, edge, boundary)\n"
     "2. Identify any missing scenarios\n"
     "3. Flag any test cases that are duplicate or vague\n"
     "4. Give a Coverage Score out of 10\n"
     "5. List any additional test cases if coverage score is below 8\n"
     "Be specific and professional."),
    ("human",
     "User Story:\n{user_story}\n\n"
     "Generated Test Cases:\n{test_cases}")
])

review_chain = review_prompt | llm | StrOutputParser()

def reviewer_node(state: TCGState) -> TCGState:
    print("[Node 3] Reviewing test cases...")
    review = review_chain.invoke({
        "user_story": state["user_story"],
        "test_cases": state["test_cases"]
    })
    final = (
        f"# TEST CASES\n\n{state['test_cases']}\n\n"
        f"---\n\n# QA LEAD REVIEW\n\n{review}"
    )
    return {**state, "review": review, "final_output": final}


## Build the LangGraph

In [9]:
graph = StateGraph(TCGState)

graph.add_node("story_analyzer",      story_analyzer_node)
graph.add_node("test_case_generator", test_case_generator_node)
graph.add_node("reviewer",            reviewer_node)

graph.set_entry_point("story_analyzer")
graph.add_edge("story_analyzer",      "test_case_generator")
graph.add_edge("test_case_generator", "reviewer")
graph.add_edge("reviewer",            END)

app = graph.compile()
print("Graph compiled successfully")
print("Flow: story_analyzer → test_case_generator → reviewer → END")


Graph compiled successfully
Flow: story_analyzer → test_case_generator → reviewer → END


## Run the Agent
Change `user_story` below to any story from your project. Banking and insurance examples are provided as comments.

In [10]:
# ── Example User Stories (uncomment one or write your own) ──────────────

user_story = """
As a bank customer,
I want to transfer funds between my accounts (savings and current)
So that I can manage my money efficiently.

Acceptance Criteria:
- Transfer amount must be greater than zero
- Source account must have sufficient balance
- Transfer limit per day is $10,000
- Confirmation SMS must be sent after successful transfer
- Transaction must appear in account history within 5 minutes
"""

# user_story = """
# As an insurance agent,
# I want to submit a new motor insurance claim on behalf of the customer
# So that the claim can be processed within the agreed SLA of 48 hours.
# Acceptance Criteria:
# - All mandatory fields (policy number, incident date, damage type) must be filled
# - Supporting documents (photos, police report) must be uploaded
# - Claim ID must be generated and emailed to customer
# - Claim status must be visible in customer portal
# """

print("User story set. Running agent...")
print("-" * 50)

initial_state: TCGState = {
    "user_story": user_story.strip(),
    "analysis": "",
    "test_cases": "",
    "review": "",
    "final_output": "",
}

result = app.invoke(initial_state)
print("\nAgent completed.")


User story set. Running agent...
--------------------------------------------------
[Node 1] Analyzing user story...
[Node 2] Generating test cases...
[Node 3] Reviewing test cases...

Agent completed.


## Results

In [11]:
print("=" * 60)
print("STORY ANALYSIS")
print("=" * 60)
print(result["analysis"])


STORY ANALYSIS
Here's the extracted information:

**Actor(s)**: Bank customer (the user performing the action)

**Action(s)**:
1. Initiate fund transfer between savings and current accounts.
2. Verify sufficient balance in source account.
3. Enter transfer amount.

**Goal**: The expected outcome is a successful fund transfer, allowing the bank customer to manage their money efficiently.

**Preconditions**:
1. The bank customer has at least one savings account and one current account.
2. The bank customer's accounts are linked to the banking system.
3. The bank customer has a valid phone number for SMS notifications.

**Business Rules**:
1. **Transfer Limit**: The total daily transfer amount cannot exceed $10,000.
2. **Balance Verification**: The source account must have sufficient balance to cover the transfer amount.
3. **Transaction Timing**: A transaction must be recorded in the account history within 5 minutes of successful transfer completion.

Note: These business rules are speci

In [12]:
print("=" * 60)
print("GENERATED TEST CASES")
print("=" * 60)
print(result["test_cases"])


GENERATED TEST CASES
Based on the user story analysis, I've generated comprehensive test cases covering positive, negative, and edge scenarios. Here are six test cases:

**Test Case 1: Successful Fund Transfer with Valid Amount**

* Test Case ID: TC_001
* Type: Positive
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) and destination account (current).
	4. Enter a valid transfer amount ($5,000).
	5. Confirm the transaction details.
* Expected Result: A confirmation SMS is sent after successful transfer, and the transaction appears in the account history within 5 minutes.
* Priority: High



In [13]:
print("=" * 60)
print("QA LEAD REVIEW")
print("=" * 60)
print(result["review"])


QA LEAD REVIEW
**Coverage Review**

After reviewing the generated test cases, I have identified the following coverage:

1. **Positive Scenarios**: All six test cases cover positive scenarios for successful fund transfers with valid amounts, insufficient balances in source accounts, exceeding daily transfer limits, and large transfer amounts.
2. **Negative Scenarios**: The test cases cover negative scenarios for invalid phone numbers for SMS notifications and insufficient balance in the source account.
3. **Edge Scenarios**: Test Case 6 covers an edge scenario by testing a large transfer amount.

However, there are some areas that need further coverage:

* **Boundary Values**: There is no test case to cover boundary values such as zero or negative amounts.
* **Error Handling**: While the system displays error messages for insufficient balance and exceeding daily limits, it would be beneficial to have additional test cases to ensure that these errors are handled correctly.

**Missing Sc

In [14]:
# Pretty display in notebook
display(Markdown(result["final_output"]))


# TEST CASES

Based on the user story analysis, I've generated comprehensive test cases covering positive, negative, and edge scenarios. Here are six test cases:

**Test Case 1: Successful Fund Transfer with Valid Amount**

* Test Case ID: TC_001
* Type: Positive
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) and destination account (current).
	4. Enter a valid transfer amount ($5,000).
	5. Confirm the transaction details.
* Expected Result: A confirmation SMS is sent after successful transfer, and the transaction appears in the account history within 5 minutes.
* Priority: High

**Test Case 2: Insufficient Balance in Source Account**

* Test Case ID: TC_002
* Type: Negative
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) with an insufficient balance ($0).
	4. Enter a valid transfer amount ($5,000).
	5. Confirm the transaction details.
* Expected Result: The system displays an error message indicating insufficient balance in the source account.
* Priority: Medium

**Test Case 3: Exceeding Daily Transfer Limit**

* Test Case ID: TC_003
* Type: Negative
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) with sufficient balance ($10,000).
	4. Enter a transfer amount exceeding the daily limit ($15,000).
	5. Confirm the transaction details.
* Expected Result: The system displays an error message indicating the daily transfer limit has been exceeded.
* Priority: High

**Test Case 4: Invalid Phone Number for SMS Notifications**

* Test Case ID: TC_004
* Type: Negative
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) with sufficient balance ($5,000).
	4. Enter an invalid phone number for SMS notifications.
	5. Confirm the transaction details.
* Expected Result: The system displays an error message indicating an invalid phone number for SMS notifications.
* Priority: Medium

**Test Case 5: Successful Fund Transfer with Zero Amount**

* Test Case ID: TC_005
* Type: Positive
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) with sufficient balance ($5,000).
	4. Enter a transfer amount of zero ($0).
	5. Confirm the transaction details.
* Expected Result: A confirmation SMS is sent after successful transfer, and the transaction appears in the account history within 5 minutes.
* Priority: Medium

**Test Case 6: Edge Scenario - Large Transfer Amount**

* Test Case ID: TC_006
* Type: Positive
* Preconditions:
	+ The bank customer has at least one savings account and one current account.
	+ The bank customer's accounts are linked to the banking system.
	+ The bank customer has a valid phone number for SMS notifications.
	+ The transfer amount is greater than zero.
* Test Steps:
	1. Log in to the online banking system using a valid username and password.
	2. Navigate to the "Transfer Funds" section.
	3. Select the source account (savings) with sufficient balance ($10,000).
	4. Enter a large transfer amount ($20,000).
	5. Confirm the transaction details.
* Expected Result: The system displays an error message indicating the daily transfer limit has been exceeded.
* Priority: High

These test cases cover various scenarios to ensure that the fund transfer functionality is working correctly and securely.

---

# QA LEAD REVIEW

**Coverage Review**

After reviewing the generated test cases, I have identified the following coverage:

1. **Positive Scenarios**: All six test cases cover positive scenarios for successful fund transfers with valid amounts, insufficient balances in source accounts, exceeding daily transfer limits, and large transfer amounts.
2. **Negative Scenarios**: The test cases cover negative scenarios for invalid phone numbers for SMS notifications and insufficient balance in the source account.
3. **Edge Scenarios**: Test Case 6 covers an edge scenario by testing a large transfer amount.

However, there are some areas that need further coverage:

* **Boundary Values**: There is no test case to cover boundary values such as zero or negative amounts.
* **Error Handling**: While the system displays error messages for insufficient balance and exceeding daily limits, it would be beneficial to have additional test cases to ensure that these errors are handled correctly.

**Missing Scenarios**

Based on the coverage review, I recommend adding the following test cases:

1. **Test Case 7: Zero Amount Transfer**
	* Preconditions:
		+ The bank customer has at least one savings account and one current account.
		+ The bank customer's accounts are linked to the banking system.
		+ The bank customer has a valid phone number for SMS notifications.
	* Test Steps:
		1. Log in to the online banking system using a valid username and password.
		2. Navigate to the "Transfer Funds" section.
		3. Select the source account (savings) with sufficient balance ($5,000).
		4. Enter a transfer amount of zero ($0).
		5. Confirm the transaction details.
	* Expected Result: The system displays an error message indicating that the transfer amount is invalid.

2. **Test Case 8: Negative Amount Transfer**
	* Preconditions:
		+ The bank customer has at least one savings account and one current account.
		+ The bank customer's accounts are linked to the banking system.
		+ The bank customer has a valid phone number for SMS notifications.
	* Test Steps:
		1. Log in to the online banking system using a valid username and password.
		2. Navigate to the "Transfer Funds" section.
		3. Select the source account (savings) with sufficient balance ($5,000).
		4. Enter a negative transfer amount (-$100).
		5. Confirm the transaction details.
	* Expected Result: The system displays an error message indicating that the transfer amount is invalid.

**Duplicate or Vague Test Cases**

After reviewing the test cases, I did not identify any duplicate or vague test cases. However, it would be beneficial to review and refine the test cases to ensure they are clear and concise.

**Coverage Score**

Based on the coverage review, I would give a Coverage Score of 8 out of 10. The test cases cover most scenarios, but there is still room for improvement by adding more boundary value tests and error handling scenarios.